Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1)Data Format & Digital Twin Justification

### Format Chosen: JSON (JavaScript Object Notation)

Justification: JSON is utilized because it is a lightweight, language-independent data-interchange format. It is natively supported by both REST HTTP (via application/json headers) and MQTT payloads. Its key-value structure allows the digital twin backend to easily parse and map incoming metrics to their corresponding virtual object properties.

### Data Types & Relevance:

Numbers (Environmental Sensors): Simulates IoT sensors in a server rack (Temperature, Humidity, Power). In a digital twin, this numerical data is crucial for asynchronous state monitoring (via HTTP) to predict overheating or track energy efficiency.

Coordinates (GPS Telemetry): Simulates automated delivery bots moving around a campus. In a digital twin, streaming coordinates (via MQTT) allows for real-time spatial mapping, route optimization, and collision avoidance of physical assets.

## 2)MQTT Protocol Justification & Routing

### Protocol Chosen: MQTT (Message Queuing Telemetry Transport)

Justification: MQTT is utilized for streaming our GPS Coordinate data because it is explicitly designed for high-frequency, continuous IoT telemetry. Unlike HTTP REST which opens and closes a connection for every request, MQTT maintains a lightweight, persistent connection to a broker.

Protocol Route/Interface: We are utilizing a public MQTT broker (broker.hivemq.com) over port 1883. The digital twin publishes and subscribes to a specific interface topic (utp/digitaltwin/telemetry/coordinates). The use of loop_start() ensures the network traffic flow runs in a threaded background process, allowing asynchronous data reception without blocking the main application logic.

In [ ]:
%pip install flask requests
%pip install paho-mqtt

1) create random data source


In [8]:
import random
from datetime import datetime
import json

def generate_number_data(source_id):
    return {
        "source_id": source_id,
        "data_type": "Environmental_Numbers",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "temperature_c": round(random.uniform(22.0, 28.0), 2),
            "humidity_pct": round(random.uniform(45.0, 55.0), 2),
            "power_usage_kw": round(random.uniform(1.2, 3.8), 2)
        }
    }

def generate_coordinate_data(source_id):
    base_lat = 4.385  
    base_lon = 100.979 
    
    return {
        "source_id": source_id,
        "data_type": "GPS_Coordinates",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "latitude": round(base_lat + random.uniform(-0.005, 0.005), 6),
            "longitude": round(base_lon + random.uniform(-0.005, 0.005), 6),
            "speed_kmh": round(random.uniform(0.0, 15.0), 1)
        }
    }


#test
print("=== Asynchronous Update Sources (Numbers) ===")
http_source_1 = number_data("Sensor_A")
http_source_2 = number_data("Sensor_B")
print(json.dumps(http_source_1, indent=2))
print(json.dumps(http_source_2, indent=2))

print("\n=== Stream Update Sources (Coordinates) ===")
mqtt_source_1 = coordinate_data("Bot_1")
mqtt_source_2 = coordinate_data("Bot_2")
print(json.dumps(mqtt_source_1, indent=2))
print(json.dumps(mqtt_source_2, indent=2))

=== Asynchronous Update Sources (Numbers) ===
{
  "source_id": "Sensor_A",
  "data_type": "Environmental_Numbers",
  "timestamp": "2026-07-04T08:51:39.729945",
  "payload": {
    "temperature_c": 25.17,
    "humidity_pct": 53.56,
    "power_usage_kw": 1.68
  }
}
{
  "source_id": "Sensor_B",
  "data_type": "Environmental_Numbers",
  "timestamp": "2026-07-04T08:51:39.729945",
  "payload": {
    "temperature_c": 27.33,
    "humidity_pct": 54.96,
    "power_usage_kw": 2.01
  }
}

=== Stream Update Sources (Coordinates) ===
{
  "source_id": "Bot_1",
  "data_type": "GPS_Coordinates",
  "timestamp": "2026-07-04T08:51:39.729945",
  "payload": {
    "latitude": 4.385715,
    "longitude": 100.976299,
    "speed_kmh": 6.7
  }
}
{
  "source_id": "Bot_2",
  "data_type": "GPS_Coordinates",
  "timestamp": "2026-07-04T08:51:39.729945",
  "payload": {
    "latitude": 4.38625,
    "longitude": 100.980372,
    "speed_kmh": 6.3
  }
}


2) create HHTP endpoint

In [11]:
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)

http_received_storage = []

@app.route('/api/twin/update', methods=['POST'])
def receive_twin_update():
    if not request.is_json:
        return jsonify({"error": "Payload must be JSON"}), 400
    
    data = request.get_json()
    
    http_received_storage.append(data)
    
    print(f"\n[HTTP SERVER] Successfully received data from Source: {data.get('source_id')}")
    print(f"[HTTP SERVER] Payload: {data}")
    
    return jsonify({"status": "success", "message": "Data logged to Digital Twin server"}), 200

def start_flask_server():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=start_flask_server, daemon=True)
server_thread.start()

print("Server thread started. Listening on http://127.0.0.1:5000/api/twin/update")

Server thread started. Listening on http://127.0.0.1:5000/api/twin/update
 * Serving Flask app '__main__'


 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


3) create MQTT endpoint

In [3]:
import paho.mqtt.client as mqtt
import json
import time

mqtt_received_storage = []

MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "utp/digitaltwin/telemetry/coordinates"

def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        print(f"[MQTT SERVER] Successfully connected to {MQTT_BROKER}.")
        client.subscribe(MQTT_TOPIC)
    else:
        print(f"[MQTT ERROR] Connection failed with code {reason_code}")

def on_message(client, userdata, msg):
    payload = json.loads(msg.payload.decode('utf-8'))
    
    mqtt_received_storage.append(payload)
    
    print(f"\n[MQTT SUBSCRIBER] Received Stream Update from Source: {payload.get('source_id')}")
    print(f"[MQTT SUBSCRIBER] Topic: {msg.topic}")
    print(f"[MQTT SUBSCRIBER] Payload: {payload}")


mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="utp_dt_jupyter_client")
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message

mqtt_client.connect(MQTT_BROKER, MQTT_PORT)

mqtt_client.loop_start() 

print(f"MQTT Client is initializing... waiting for connection callback.")

MQTT Client is initializing... waiting for connection callback.


[MQTT SERVER] Successfully connected to broker.hivemq.com.


In [9]:
# Quick Test: Publish a single coordinate dataset to the broker
test_bot_data = generate_coordinate_data("Bot_1")

# Publish the JSON string to our topic
mqtt_client.publish(MQTT_TOPIC, json.dumps(test_bot_data))


[MQTT SUBSCRIBER] Received Stream Update from Source: Bot_1
[MQTT SUBSCRIBER] Topic: utp/digitaltwin/telemetry/coordinates
[MQTT SUBSCRIBER] Payload: {'source_id': 'Bot_1', 'data_type': 'GPS_Coordinates', 'timestamp': '2026-07-04T08:51:46.703482', 'payload': {'latitude': 4.387129, 'longitude': 100.979596, 'speed_kmh': 9.7}}


In [ ]:
import concurrent.futures
import time
import requests
import json

# 1. Clear previous test data from the server storage lists to ensure a clean run
http_received_storage.clear()
mqtt_received_storage.clear()

print("=== Generating Digital Twin Data ===")
# 2. Generate the 4 distinct data sources
http_payload_1 = generate_number_data("Server_Rack_Sensor_A")
http_payload_2 = generate_coordinate_data("Server_Rack_Sensor_B")

mqtt_payload_1 = generate_coordinate_data("UTP_Express_Laundry_Bot_1")
mqtt_payload_2 = generate_number_data("UTP_Express_Laundry_Bot_2")
print("Data generated successfully for all 4 sources.\n")

# 3. Define the transmission wrapper functions for the ThreadPool
def trigger_http_protocol(payload):
    requests.post("http://127.0.0.1:5000/api/twin/update", json=payload)
    return f"[HTTP TRIGGER] Transmitted data for {payload['source_id']}"

def trigger_mqtt_protocol(payload):
    mqtt_client.publish(MQTT_TOPIC, json.dumps(payload))
    return f"[MQTT TRIGGER] Transmitted data for {payload['source_id']}"

print("=== Initiating Parallel Transmission ===")
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(trigger_http_protocol, http_payload_1),
        executor.submit(trigger_http_protocol, http_payload_2),
        executor.submit(trigger_mqtt_protocol, mqtt_payload_1),
        executor.submit(trigger_mqtt_protocol, mqtt_payload_2)
    ]
    
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

print("\nWaiting for asynchronous network traffic to settle...")
time.sleep(2)

print("\n=== Verifying Protocol Correctness ===")
try:
    # HTTP Verification
    assert http_payload_1 in http_received_storage, "HTTP Source 1 missing or corrupted!"
    assert http_payload_2 in http_received_storage, "HTTP Source 2 missing or corrupted!"
    print("✅ HTTP REST: Verified send & receive for asynchronous data updates.")

    # MQTT Verification
    assert mqtt_payload_1 in mqtt_received_storage, "MQTT Source 1 missing or corrupted!"
    assert mqtt_payload_2 in mqtt_received_storage, "MQTT Source 2 missing or corrupted!"
    print("✅ MQTT Stream: Verified send & receive for continuous data streams.")
    
    print("\n🎉 ALL PROTOCOLS VERIFIED SUCCESSFULLY.")

except AssertionError as e:
    print(f"❌ Verification Failed: {e}")


127.0.0.1 - - [04/Jul/2026 09:02:02] "POST /api/twin/update HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 09:02:02] "POST /api/twin/update HTTP/1.1" 200 -


=== Generating Digital Twin Data ===
Data generated successfully for all 4 sources.

=== Initiating Parallel Transmission ===

[HTTP SERVER] Successfully received data from Source: Server_Rack_Sensor_A
[HTTP SERVER] Payload: {'source_id': 'Server_Rack_Sensor_A', 'data_type': 'Environmental_Numbers', 'timestamp': '2026-07-04T09:02:02.849839', 'payload': {'temperature_c': 23.2, 'humidity_pct': 53.44, 'power_usage_kw': 1.21}}
[MQTT TRIGGER] Transmitted data for UTP_Express_Laundry_Bot_1
[MQTT TRIGGER] Transmitted data for UTP_Express_Laundry_Bot_2

[HTTP SERVER] Successfully received data from Source: Server_Rack_Sensor_B
[HTTP SERVER] Payload: {'source_id': 'Server_Rack_Sensor_B', 'data_type': 'GPS_Coordinates', 'timestamp': '2026-07-04T09:02:02.849839', 'payload': {'latitude': 4.387181, 'longitude': 100.981877, 'speed_kmh': 7.4}}
[HTTP TRIGGER] Transmitted data for Server_Rack_Sensor_A
[HTTP TRIGGER] Transmitted data for Server_Rack_Sensor_B

Waiting for asynchronous network traffic to 